#### Visão Geral
##### Schema : silver
##### Table : case_cadastro_produtos

| Detalhe | Informação |
|---------|------------|
| Criado Originalmente Por | Wellikiandre Bosich |
| Tabela de Dados de Saída | `{environment}.silver.case_cadastro_produtos` |
| Origem Fonte de Dados de Entrada | Camada bronze |
| Destino Fonte de Dados de Saída | Camada silver |

#### Histórico

| Data       | Desenvolvido Por         | Motivo                                         |
|:----------:|--------------------------|-----------------------------------------------|
| 04/06/2026 | Wellikiandre Bosich    | Criação do notebook e tratamento/parse do dump de API de produtos para a Silver. |

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
sistema = 'case'
table_name = 'cadastro_produtos_api_dump'
output_table_name = 'cadastro_produtos'
input_path = f"{var_bronze}/{sistema}/{table_name}/data"
output_path_data = f"{var_silver}/{sistema}/{output_table_name}/data"
table_name_schema = f'{var_environment}.{var_silver_schema}.{sistema}_{output_table_name}'

In [ ]:
from pyspark.sql.functions import col, to_timestamp, row_number, coalesce, lit
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType

df_bronze = spark.read.format("delta").load(input_path)

# Janela para deduplicação pelo produto mais recente atualizado
window_spec = Window.partitionBy("product_id").orderBy(col("updated_at").desc())

df_clean = (
    df_bronze
    .withColumn("updated_at", to_timestamp(col("updated_at")))
    .withColumn("rn", row_number().over(window_spec))
    .filter("rn = 1")
    .drop("rn")
    .withColumn("list_price", col("list_price").cast(DecimalType(10, 2)))
    .select(
        col("product_id").cast("string").alias("id_produto"),
        col("product_name").cast("string").alias("nome_produto"),
        coalesce(col("product_category"), lit("Outros")).cast("string").alias("categoria_produto"),
        col("product_subcategory").cast("string").alias("subcategoria_produto"),
        col("product_status").cast("string").alias("status_produto"),
        col("list_price").alias("preco_tabela"),
        col("currency").cast("string").alias("moeda"),
        col("family").cast("string").alias("familia_produto"),
        col("updated_at").alias("data_atualizacao")
    )
    .filter(col("id_produto").isNotNull())
)

In [ ]:
process_data(
    df_write=df_clean,
    tipo_carga='delta',
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    chave_clusterby=['categoria_produto'],
    chave_upsert='id_produto'
)